# P2 CHAIN B — library inventory (OOP → ADT → S&S → JSON)

Closed-book mega-question. Reference voice: `POOP.ipynb`, `ADT.ipynb`, `S&S.ipynb`. **Do not open them.**

**Rules**
- Parts depend on earlier objects in the same kernel (`stock` → linked list → hash/BST → sorted keys → JSON).
- Use double-underscore privates + getters; child must not read parent `__attrs` directly.
- **Checkpoints** are encapsulated at the **bottom** of this notebook — run them only after you finish a part (or the whole chain). Do not treat them as a writing guide.
- **No answer keys in this notebook.** Mark against boilerplates / archive answers only after attempting.

**Domain:** library resources — print books and digital resources.


---
## Part B1 — OOP hierarchy (12 min)

Implement:

- `Resource(resource_id, title)` with private `__resource_id`, `__title`, getters, `set_title`, and `describe()` → `Resource {id}: {title}`
- `DigitalResource(resource_id, title, filesize_mb)` inheriting from `Resource`: call `super().__init__`, store private `__filesize_mb` as `int`, override `describe()` using **getters** for parent fields → `Digital Resource {id}: {title} ({n} MB)`

Build `stock`: a Python list of at least 6 mixed Resource / DigitalResource instances. Print each `.describe()`.


In [ ]:
# Part B1 — YOUR ATTEMPT

stock = []
# classes + populate stock


---
## Part B2 — LinkedList of resource IDs (15 min)

`Node` (data + right link, getters/setters) and `LinkedList` with `__start`, `is_empty`, `insert_end`, `search` (Node or None), `delete` (**including head case**).

Insert every `resource_id` from `stock` via `insert_end` into `catalogue = LinkedList()`.

Then: `delete` the **first** id you inserted (head), `search` a mid id (found), `search(-1)` → None, `delete` a missing id (no crash).


In [ ]:
# Part B2 — YOUR ATTEMPT

catalogue = None  # LinkedList()
# Node, LinkedList, drive from stock


---
## Part B3 — HashTable + BST report (15 min)

1. `HashTable(N)` separate chaining: `self.__arr = [LinkedList() for _ in range(N)]` — never `[LinkedList()] * N`. Methods: `hash_function`, `insert`, `search` (bool). Insert all ids currently in `catalogue`.
2. `TreeNode` + `BST` with `insert` (`<=` left, `>` right) and `in_order` / `pre_order` / `post_order` each taking a **node**. Insert same ids. Print in-order.

Use `N = 7`. Keep `ht` and `bst` in the kernel. Prefer a `walk_ids()` helper or equivalent on the list.


In [ ]:
# Part B3 — YOUR ATTEMPT

ht = None
bst = None
# HashTable, TreeNode, BST + load from catalogue


---
## Part B4 — sort + binary search on extracted keys (12 min)

Build `keys` (list of ints) from the catalogue.

1. **Insertion sort** *or* `partition` + `quickSort` (final pivot↔rightmark swap). Sort a **copy** into `sorted_keys`.
2. Recursive `BinarySearch(Arr, FindValue, Low, High)` — `Low > High` → `-1`; recurse with `middle ± 1` and **return** the call.

Demo: search existing id and missing id on `sorted_keys`.


In [ ]:
# Part B4 — YOUR ATTEMPT

keys = []
sorted_keys = []
# extract, sort, BinarySearch demos


---
## Part B5 — JSON round-trip (8 min)

Build a JSON-serialisable list of dicts from `stock`, e.g.
`{"id": ..., "title": ..., "kind": "print"|"digital", "filesize_mb": ...|null}`.

- `save_catalogue(filename, data)` — `json.dump`, utf-8
- `load_catalogue(filename)` — `json.load`

Write `library_out.json`, load it back, assert same length.

When finished practising, delete `library_out.json` so it does not linger as a trail.


In [ ]:
# Part B5 — YOUR ATTEMPT
import json

# save_catalogue / load_catalogue / round-trip


---
# Checkpoints (encapsulated)

Run **after** attempting. Edit `PARTS` below to choose what to mark. These cells are not a guide while writing.


In [ ]:
# CHECKPOINTS — Chain B (run after attempts)
# Set which parts to mark, then run this cell once.

PARTS = ("B1", "B2", "B3", "B4", "B5")  # e.g. ("B1", "B2") while still climbing


def _rid(obj):
    if hasattr(obj, "get_resource_id"):
        return obj.get_resource_id()
    return obj.get_id()


def _check_b1():
    g = globals()
    if "Resource" not in g or "DigitalResource" not in g:
        return False, "missing class"
    r = Resource(1, "Alpha")
    d = DigitalResource(2, "Beta", "12")
    if "Resource 1: Alpha" not in r.describe():
        return False, f"Resource.describe odd: {r.describe()!r}"
    if "Digital Resource 2: Beta" not in d.describe() or "12" not in d.describe():
        return False, f"DigitalResource.describe odd: {d.describe()!r}"
    if not isinstance(d, Resource):
        return False, "DigitalResource should be subclass of Resource"
    if len(stock) < 6:
        return False, f"stock length {len(stock)} < 6"
    return True, "ok"


def _check_b2():
    first_id = _rid(stock[0])
    mid_id = _rid(stock[2])
    if catalogue is None or catalogue.is_empty():
        return False, "catalogue empty/missing"
    if catalogue.search(first_id) is not None:
        return False, "head id still present after required delete"
    if catalogue.search(mid_id) is None:
        return False, "mid id missing — insert_end/search broken?"
    if catalogue.search(-1) is not None:
        return False, "search(-1) should be None"
    return True, "ok"


def _check_b3():
    mid_id = _rid(stock[2])
    if ht is None or bst is None:
        return False, "ht or bst missing"
    if not ht.search(mid_id):
        return False, "hash search failed for mid id"
    if hasattr(bst, "in_order") and hasattr(bst, "get_root"):
        bst.in_order(bst.get_root())
    return True, "ok"


def _check_b4():
    if sorted_keys != sorted(sorted_keys):
        return False, "sorted_keys not sorted"
    if len(sorted_keys) < 5:
        return False, "too few keys"
    target = sorted_keys[len(sorted_keys) // 2]
    idx = BinarySearch(sorted_keys, target, 0, len(sorted_keys) - 1)
    if idx < 0 or sorted_keys[idx] != target:
        return False, "BinarySearch miss on existing value"
    if BinarySearch(sorted_keys, -999, 0, len(sorted_keys) - 1) != -1:
        return False, "BinarySearch should return -1 for absent"
    return True, "ok"


def _check_b5():
    from pathlib import Path
    import json
    p = Path("library_out.json")
    if not p.exists():
        return False, "library_out.json missing"
    data = json.loads(p.read_text(encoding="utf-8"))
    if len(data) != len(stock):
        return False, f"json len {len(data)} != stock {len(stock)}"
    if not isinstance(data[0], dict) or "id" not in data[0]:
        return False, "records should be dicts with id"
    return True, "ok"


_CHECKS = {
    "B1": _check_b1,
    "B2": _check_b2,
    "B3": _check_b3,
    "B4": _check_b4,
    "B5": _check_b5,
}

for part in PARTS:
    try:
        ok, detail = _CHECKS[part]()
    except Exception as e:
        ok, detail = False, str(e)
    print(f"{part}: {'PASS' if ok else 'FAIL — ' + detail}")
